# 2. Stats model (supervised baseline)

Predict the next pitch type **from the game situation only**: count, batter side, base/out state, score, inning, times through the order, and the previous pitches in the at-bat.

Evaluation: train on older games, test on the most recent 20% of games (like scouting his next outing). Compared against two naive baselines: always predicting his overall mix, and predicting his mix for the current count.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # so cache/ and reports/ live at the repo root
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pitchscout.workspace import Config, get_pitches, get_clips
from pitchscout import plotting as P

# DEMO=True runs on synthetic data with no network (numbers are NOT real).
DEMO = os.environ.get("PITCHSCOUT_DEMO") == "1"
cfg = Config(pitcher="Mason Miller", start="2024-03-28", end="2025-09-28", n_clips=600, demo=DEMO)
if DEMO:
    print("DEMO MODE: synthetic data, results do not describe a real pitcher")

In [ ]:
from pitchscout import baseline, features
df, throws = get_pitches(cfg)
X = features.situation_features(df)
X.head()

In [ ]:
base = baseline.fit_baseline(df)
m = base.metrics
table = pd.DataFrame({k: m[k] for k in ["naive_mix", "count_only", "stats_model"]}).T
table.index = ["Overall mix", "Mix by count", "Stats model (LightGBM)"]
table["family"] = ["baseline", "baseline", "supervised"]
print(f"train {m['n_train']:,} pitches / test {m['n_test']:,} pitches")
table

In [ ]:
P.compare_models(table, "Stats layer on held-out games"); plt.show()

## What the stats model relies on

In [ ]:
P.hbar(base.importance.head(12), "Feature importance (share of split gain)"); plt.show()

## Calibration

When the model says 70% fastball, is it a fastball about 70% of the time? This matters because later layers use these probabilities as their prior.

In [ ]:
from sklearn.calibration import calibration_curve
y_te = df.pitch_type.to_numpy()[base.test_idx]
proba = base.model.predict_proba(X.iloc[base.test_idx])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for k, c in enumerate(base.classes):
    frac, mean = calibration_curve(y_te == c, proba[:, k], n_bins=8, strategy="quantile")
    ax.plot(mean, frac, "o-", color=P.pitch_colors(base.classes)[c], label=P.name(c), ms=4)
ax.plot([0, 1], [0, 1], "--", color="#c3c2b7"); ax.set_xlabel("predicted"); ax.set_ylabel("observed")
ax.legend(frameon=False); ax.set_title("Calibration on held-out games", loc="left"); plt.show()